# 02 — Data pipeline: load → validate → clean → split → preprocess

Executed by `make train` (and `make data`). Produces:

| Artifact | Purpose |
|---|---|
| `reports/data_validation.json` | result of every data check |
| `data/processed/{train,val,test}.csv` | stratified 70/15/15 split, leakage-free features + target |
| `models/preprocessor.joblib` | preprocessing pipeline **fitted on the training set only** |
| `models/feature_config.json` | feature lists, target, seed — reused by training, API and dashboard |
| `reports/split_summary.json` | sizes and churn rate of each split |

Rule: the test set is created here and **not touched** until the final evaluation.

In [1]:
import json
import os
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def find_root(start=None):
    """Project root = first parent folder containing the Makefile (works from notebooks/ or via make)."""
    p = Path(start or Path.cwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "Makefile").exists():
            return cand
    return p

ROOT = find_root()
RAW_PATH = Path(os.environ.get("CHURN_DATA_PATH", ROOT / "data" / "raw" / "Telco_customer_churn.csv"))

PROCESSED_DIR = ROOT / "data" / "processed"
MODELS_DIR = ROOT / "models"
REPORTS_DIR = ROOT / "reports"
for d in (PROCESSED_DIR, MODELS_DIR, REPORTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
print("Project root:", ROOT)
print("Data file   :", RAW_PATH)

Project root: C:\Users\lenovo\Desktop\churn-mlops
Data file   : C:\Users\lenovo\Desktop\churn-mlops\data\raw\Telco_customer_churn.csv


## 1. Configuration — which columns are used, and why

Everything column-related lives in this one cell.

In [2]:
ID_COL = "CustomerID"
TARGET = "Churn Value"

# Leakage: unavailable at prediction time or computed from the outcome
LEAKAGE_COLS = ["Churn Label", "Churn Score", "Churn Reason", "CLTV"]
# Identifiers / constants without predictive value
NON_INFORMATIVE_COLS = ["Count", "Country", "State", "Lat Long"]
# High-cardinality geography: easy to overfit with 7k rows (revisit later as an experiment)
GEOGRAPHY_COLS = ["City", "Zip Code", "Latitude", "Longitude"]
# Dropped by choice: no business reason to score customers on a sensitive attribute
OPTIONAL_DROP_COLS = ["Gender"]

NUMERIC = ["Tenure Months", "Monthly Charges", "Total Charges"]
CATEGORICAL = [
    "Senior Citizen", "Partner", "Dependents", "Phone Service", "Multiple Lines",
    "Internet Service", "Online Security", "Online Backup", "Device Protection", "Tech Support",
    "Streaming TV", "Streaming Movies", "Contract", "Paperless Billing", "Payment Method",
]
FEATURES = NUMERIC + CATEGORICAL

excluded = set(LEAKAGE_COLS + NON_INFORMATIVE_COLS + GEOGRAPHY_COLS + OPTIONAL_DROP_COLS)
assert not (set(FEATURES) & excluded), "A leaky or excluded column slipped into FEATURES"
print(f"{len(FEATURES)} features ({len(NUMERIC)} numeric, {len(CATEGORICAL)} categorical)")

18 features (3 numeric, 15 categorical)


## 2. Load

In [4]:
df = pd.read_csv(RAW_PATH, sep=";")
df.columns = df.columns.str.strip()
df = df.rename(columns={"Monthly Charge": "Monthly Charges"})   # tolerate the shorter name

for col in df.columns:   # strip stray whitespace in text columns
    if pd.api.types.is_string_dtype(df[col]) and not pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].str.strip()

print("Loaded:", df.shape)

required = [ID_COL, TARGET] + FEATURES
missing_cols = [c for c in required if c not in df.columns]
if missing_cols:
    raise KeyError(f"Missing expected columns: {missing_cols}\nColumns found: {list(df.columns)}")

Loaded: (7043, 33)


In [6]:
df["Monthly Charges"] = (
    df["Monthly Charges"]
    .astype(str)
    .str.replace(",", ".", regex=False)
)

df["Monthly Charges"] = pd.to_numeric(
    df["Monthly Charges"],
    errors="coerce"
)

In [7]:
print(df["Monthly Charges"].dtype)

float64


In [9]:
tc = pd.to_numeric(df["Total Charges"], errors="coerce")

bad_rows = df.loc[
    tc.isna() & (df["Tenure Months"] != 0),
    ["CustomerID", "Tenure Months", "Total Charges"]
]

print("Bad rows:", len(bad_rows))
display(bad_rows.head(20))

Bad rows: 6708


,CustomerID,Tenure Months,Total Charges
0,3668-QPYBK,2,"108,15"
1,9237-HQITU,2,"151,65"
2,9305-CDSKC,8,"820,5"
3,7892-POOKP,28,"3046,05"
4,0280-XJGEX,49,"5036,3"
5,4190-MFLUW,10,"528,35"
6,8779-QRDMV,1,"39,65"
7,1066-JKSGK,1,"20,15"
8,6467-CHFZW,47,"4749,15"
9,8665-UTDHZ,1,"30,2"


In [10]:
tc = pd.to_numeric(df["Total Charges"], errors="coerce")
print("Number of NaN Total Charges:", tc.isna().sum())
print(df.loc[tc.isna(), ["CustomerID", "Tenure Months", "Total Charges"]].head())

Number of NaN Total Charges: 6719
   CustomerID  Tenure Months Total Charges
0  3668-QPYBK              2        108,15
1  9237-HQITU              2        151,65
2  9305-CDSKC              8         820,5
3  7892-POOKP             28       3046,05
4  0280-XJGEX             49        5036,3


In [11]:
df["Total Charges"] = (
    df["Total Charges"]
    .astype(str)
    .str.replace(",", ".", regex=False)
)

df["Total Charges"] = pd.to_numeric(
    df["Total Charges"],
    errors="coerce"
)

## 3. Validate

Errors stop the pipeline; warnings are recorded in the report but do not stop it.

In [12]:
checks = []

def check(name, passed, detail="", severity="error"):
    detail = "" if detail is None else str(detail)
    checks.append({"check": name, "passed": bool(passed), "severity": severity, "detail": detail})
    icon = "OK  " if passed else ("FAIL" if severity == "error" else "WARN")
    print(f"[{icon}] {name}" + (f" -> {detail}" if (detail and not passed) else ""))

BINARY_COLS = ["Senior Citizen", "Partner", "Dependents", "Phone Service", "Paperless Billing"]
ADDON_COLS = ["Online Security", "Online Backup", "Device Protection", "Tech Support", "Streaming TV", "Streaming Movies"]

# --- structure
check("enough rows (>= 1000)", len(df) >= 1000, f"{len(df)} rows")
check("CustomerID is unique", df[ID_COL].is_unique, f"{df[ID_COL].duplicated().sum()} duplicates")
check("no fully duplicated rows", not df.duplicated().any())

# --- target
check("target is binary 0/1", set(df[TARGET].unique()) <= {0, 1}, df[TARGET].unique())
if "Churn Label" in df.columns:
    expected_label = df[TARGET].map({1: "Yes", 0: "No"})
    check("Churn Label consistent with Churn Value", (df["Churn Label"] == expected_label).all())
churn_rate = df[TARGET].mean()
check("churn rate between 5% and 50%", 0.05 <= churn_rate <= 0.50, f"{churn_rate:.1%}", severity="warning")

# --- numeric columns
for col in ["Tenure Months", "Monthly Charges"]:
    check(f"{col} is numeric", pd.api.types.is_numeric_dtype(df[col]))
check("Tenure Months within 0-120", df["Tenure Months"].between(0, 120).all())
check("Monthly Charges > 0", (df["Monthly Charges"] > 0).all())

tc = pd.to_numeric(df["Total Charges"], errors="coerce")
blank_tc = tc.isna()
check("blank Total Charges only for tenure = 0", (df.loc[blank_tc, "Tenure Months"] == 0).all(),
      f"{int(blank_tc.sum())} blanks")
check("Total Charges >= 0", (tc.dropna() >= 0).all())

# --- categorical columns
other_cols = [c for c in [ID_COL, TARGET] + FEATURES if c != "Total Charges"]
check("no missing values (except Total Charges blanks)", df[other_cols].isna().sum().sum() == 0,
      df[other_cols].isna().sum()[lambda s: s > 0].to_dict())

for col in BINARY_COLS:
    bad = set(df[col].unique()) - {"Yes", "No"}
    check(f"{col} in {{Yes, No}}", not bad, bad)
bad = set(df["Multiple Lines"].unique()) - {"Yes", "No", "No phone service"}
check("Multiple Lines values", not bad, bad)
for col in ADDON_COLS:
    bad = set(df[col].unique()) - {"Yes", "No", "No internet service"}
    check(f"{col} values", not bad, bad)
bad = set(df["Contract"].unique()) - {"Month-to-month", "One year", "Two year"}
check("Contract values", not bad, bad, severity="warning")
for col in ["Internet Service", "Payment Method"]:
    check(f"{col} has <= 6 categories", df[col].nunique() <= 6, df[col].unique().tolist(), severity="warning")

# --- business consistency
no_phone = df["Phone Service"] == "No"
check("Phone Service=No implies Multiple Lines='No phone service'",
      (df.loc[no_phone, "Multiple Lines"] == "No phone service").all(), severity="warning")
no_net = df["Internet Service"] == "No"
check("Internet Service=No implies add-ons='No internet service'",
      all((df.loc[no_net, c] == "No internet service").all() for c in ADDON_COLS), severity="warning")

# --- report
report = {
    "n_rows": int(len(df)),
    "n_columns": int(df.shape[1]),
    "churn_rate": round(float(churn_rate), 4),
    "n_errors": sum(1 for c in checks if not c["passed"] and c["severity"] == "error"),
    "n_warnings": sum(1 for c in checks if not c["passed"] and c["severity"] == "warning"),
    "checks": checks,
}
(REPORTS_DIR / "data_validation.json").write_text(json.dumps(report, indent=2, default=str))

failed = [c["check"] for c in checks if not c["passed"] and c["severity"] == "error"]
if failed:
    raise ValueError(f"Data validation failed: {failed}")
print(f"\nValidation passed ({report['n_warnings']} warning(s)).")

[OK  ] enough rows (>= 1000)
[OK  ] CustomerID is unique
[OK  ] no fully duplicated rows
[OK  ] target is binary 0/1
[OK  ] Churn Label consistent with Churn Value
[OK  ] churn rate between 5% and 50%
[OK  ] Tenure Months is numeric
[OK  ] Monthly Charges is numeric
[OK  ] Tenure Months within 0-120
[OK  ] Monthly Charges > 0
[OK  ] blank Total Charges only for tenure = 0
[OK  ] Total Charges >= 0
[OK  ] no missing values (except Total Charges blanks)
[OK  ] Senior Citizen in {Yes, No}
[OK  ] Partner in {Yes, No}
[OK  ] Dependents in {Yes, No}
[OK  ] Phone Service in {Yes, No}
[OK  ] Paperless Billing in {Yes, No}
[OK  ] Multiple Lines values
[OK  ] Online Security values
[OK  ] Online Backup values
[OK  ] Device Protection values
[OK  ] Tech Support values
[OK  ] Streaming TV values
[OK  ] Streaming Movies values
[OK  ] Contract values
[OK  ] Internet Service has <= 6 categories
[OK  ] Payment Method has <= 6 categories
[OK  ] Phone Service=No implies Multiple Lines='No phone service'

## 4. Clean and build the modelling table

- `Total Charges`: blanks (tenure 0) become 0.
- Keep only `CustomerID` (for traceability), the features and the target.

In [13]:
df["Total Charges"] = pd.to_numeric(df["Total Charges"], errors="coerce").fillna(0.0)

df_model = df[[ID_COL] + FEATURES + [TARGET]].copy()
df_model[TARGET] = df_model[TARGET].astype(int)

print("Dropped columns:", sorted(set(df.columns) - set(df_model.columns)))
print("Modelling table:", df_model.shape)

Dropped columns: ['CLTV', 'Churn Label', 'Churn Reason', 'Churn Score', 'City', 'Count', 'Country', 'Gender', 'Lat Long', 'Latitude', 'Longitude', 'State', 'Zip Code']
Modelling table: (7043, 20)


## 5. Stratified train / validation / test split (70 / 15 / 15)

In [14]:
train_df, tmp_df = train_test_split(df_model, test_size=0.30, stratify=df_model[TARGET], random_state=SEED)
val_df, test_df = train_test_split(tmp_df, test_size=0.50, stratify=tmp_df[TARGET], random_state=SEED)

assert set(train_df[ID_COL]).isdisjoint(val_df[ID_COL])
assert set(train_df[ID_COL]).isdisjoint(test_df[ID_COL])
assert set(val_df[ID_COL]).isdisjoint(test_df[ID_COL])

split_summary = {
    name: {"rows": int(len(part)), "churn_rate": round(float(part[TARGET].mean()), 4)}
    for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]
}
for name, s in split_summary.items():
    print(f"{name:>5}: {s['rows']:>5} rows | churn rate {s['churn_rate']:.1%}")

train_df.to_csv(PROCESSED_DIR / "train.csv", index=False)
val_df.to_csv(PROCESSED_DIR / "val.csv", index=False)
test_df.to_csv(PROCESSED_DIR / "test.csv", index=False)
(REPORTS_DIR / "split_summary.json").write_text(json.dumps(split_summary, indent=2))

train:  4930 rows | churn rate 26.5%
  val:  1056 rows | churn rate 26.5%
 test:  1057 rows | churn rate 26.6%


182

## 6. Preprocessing pipeline

Built only with scikit-learn components, so the saved file can be loaded by the API without importing any project code.

- numeric: median imputation, standard scaling
- categorical: most-frequent imputation, one-hot encoding (binary columns become a single 0/1 column; unseen categories at inference are ignored)

It is **fitted on the training set only** to avoid leaking information from validation/test.

In [15]:
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(drop="if_binary", handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer(
    [("num", numeric_pipe, NUMERIC), ("cat", categorical_pipe, CATEGORICAL)],
    verbose_feature_names_out=False,
)
preprocessor.set_output(transform="pandas")

X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_val, y_val = val_df[FEATURES], val_df[TARGET]
X_test = test_df[FEATURES]

Xt_train = preprocessor.fit_transform(X_train)   # fit on train only
Xt_val = preprocessor.transform(X_val)
Xt_test = preprocessor.transform(X_test)

assert not Xt_train.isna().any().any() and not Xt_val.isna().any().any() and not Xt_test.isna().any().any()
assert list(Xt_train.columns) == list(Xt_val.columns) == list(Xt_test.columns)
print("Transformed shapes -> train:", Xt_train.shape, "| val:", Xt_val.shape, "| test:", Xt_test.shape)
print("First features:", list(Xt_train.columns[:8]))

Transformed shapes -> train: (4930, 39) | val: (1056, 39) | test: (1057, 39)
First features: ['Tenure Months', 'Monthly Charges', 'Total Charges', 'Senior Citizen_Yes', 'Partner_Yes', 'Dependents_Yes', 'Phone Service_Yes', 'Multiple Lines_No']


In [16]:
joblib.dump(preprocessor, MODELS_DIR / "preprocessor.joblib")

feature_config = {
    "target": TARGET,
    "id_col": ID_COL,
    "numeric": NUMERIC,
    "categorical": CATEGORICAL,
    "features": FEATURES,
    "transformed_feature_names": list(Xt_train.columns),
    "excluded_columns": {
        "leakage": LEAKAGE_COLS,
        "non_informative": NON_INFORMATIVE_COLS,
        "geography": GEOGRAPHY_COLS,
        "optional": OPTIONAL_DROP_COLS,
    },
    "seed": SEED,
}
(MODELS_DIR / "feature_config.json").write_text(json.dumps(feature_config, indent=2))
print("Saved preprocessor and feature config.")

Saved preprocessor and feature config.


## 7. Sanity check (not an experiment)

A quick logistic regression on the **validation** set. On this dataset a leakage-free model typically lands around 0.8–0.85 ROC-AUC.
A score near 1.0 would mean a leaky column slipped in; a score near 0.5 would mean the preprocessing is broken.
The test set is not used.

In [17]:
sanity = LogisticRegression(max_iter=1000, class_weight="balanced")
sanity.fit(Xt_train, y_train)
val_auc = roc_auc_score(y_val, sanity.predict_proba(Xt_val)[:, 1])
print(f"Sanity-check validation ROC-AUC: {val_auc:.3f}")
assert 0.60 < val_auc < 0.95, "Suspicious AUC: check for leakage or broken preprocessing"

Sanity-check validation ROC-AUC: 0.849


## 8. Summary

In [18]:
print("Pipeline finished. Artifacts:")
for p in [REPORTS_DIR / "data_validation.json", REPORTS_DIR / "split_summary.json",
          PROCESSED_DIR / "train.csv", PROCESSED_DIR / "val.csv", PROCESSED_DIR / "test.csv",
          MODELS_DIR / "preprocessor.joblib", MODELS_DIR / "feature_config.json"]:
    print(" -", p.relative_to(ROOT))

Pipeline finished. Artifacts:
 - reports\data_validation.json
 - reports\split_summary.json
 - data\processed\train.csv
 - data\processed\val.csv
 - data\processed\test.csv
 - models\preprocessor.joblib
 - models\feature_config.json
